## RoG CWQ Baseline

### Config and Load

In [43]:
from datasets import load_dataset
CWQ_DATASET_NAME = "rmanluo/RoG-cwq"
CWQ_SPLIT = "test"
CWQ_N_PILOT = 50
CWQ_MAX_HOPS = 4   # As CWQ goes up to 4-hop 
CWQ_PILOT_DIR = "/kaggle/working/step1_cwq_pilot"
os.makedirs(CWQ_PILOT_DIR, exist_ok=True)

cwq_full_test = load_dataset(CWQ_DATASET_NAME, split=CWQ_SPLIT)
cwq_pilot = cwq_full_test.select(range(min(CWQ_N_PILOT, len(cwq_full_test))))
print(f"CWQ test set: {len(cwq_full_test)} questions (paper Table 6: 3,531)")
print(f"Pilot subset: {len(cwq_pilot)} questions")
cwq_pilot[0]

README.md:   0%|          | 0.00/913 [00:00<?, ?B/s]

data/train-00000-of-00018-e65d08d5970d44(…):   0%|          | 0.00/130M [00:00<?, ?B/s]

data/train-00001-of-00018-c70342c196c07d(…):   0%|          | 0.00/132M [00:00<?, ?B/s]

data/train-00002-of-00018-d52ad886cb9f05(…):   0%|          | 0.00/128M [00:00<?, ?B/s]

data/train-00003-of-00018-6dac2fb592f087(…):   0%|          | 0.00/157M [00:00<?, ?B/s]

...

data/test-00000-of-00003-e62a559c5d2b56c(…):   0%|          | 0.00/114M [00:00<?, ?B/s]

data/test-00001-of-00003-2fa9a898639e7d1(…):   0%|          | 0.00/128M [00:00<?, ?B/s]

data/test-00002-of-00003-c659cd388440c4a(…):   0%|          | 0.00/131M [00:00<?, ?B/s]

CWQ test set: 3531 questions (paper Table 6: 3,531)
Pilot subset: 50 questions


{'id': 'WebQTest-832_c334509bb5e02cacae1ba2e80c176499',
 'question': 'Lou Seal is the mascot for the team that last won the World Series when?',
 'answer': ['2014 World Series'],
 'q_entity': ['Lou Seal'],
 'a_entity': ['2014 World Series'],
 'graph': [['Mascot', 'type.type.expected_by', 'sports_team_mascot'],
  ['San Francisco Giants', 'baseball.baseball_team.team_stats', 'm.05n69q3'],
  ['San Francisco Giants', 'baseball.baseball_team.team_stats', 'm.05n6btw'],
  ['San Francisco Giants',
   'freebase.valuenotation.is_reviewed',
   'Contact webpages'],
  ['San Francisco Giants',
   'base.schemastaging.sports_team_extra.training_ground',
   'm.0k079rd'],
  ['Polo Grounds', 'sports.sports_facility.teams', 'San Francisco Giants'],
  ['m.04vy2qp',
   'sports.sports_league_draft_pick.team',
   'San Francisco Giants'],
  ['San Francisco Giants',
   'sports.sports_team.arena_stadium',
   'Seals Stadium'],
  ['m.05n6dtn', 'baseball.baseball_team_stats.team', 'San Francisco Giants'],
  ['San F

### Pilot Planning

In [44]:
cwq_pilot_planning = []
for sample in tqdm(cwq_pilot, desc="CWQ Planning (pilot)"):
    input_text = prompter.format(instruction=INSTRUCTION, message=sample["question"])
    t0 = time.time()
    raw_output = generate_seq(model, input_text, tokenizer, num_beam=N_BEAM, do_sample=True, max_new_tokens=100)
    planning_time = time.time() - t0
    rel_paths = parse_prediction(raw_output["paths"])
    cwq_pilot_planning.append({
        "id": sample["id"], "question": sample["question"],
        "q_entity": sample["q_entity"], "a_entity": sample["a_entity"],
        "graph": sample["graph"], "predicted_paths": rel_paths,
        "planning_time_sec": planning_time,
    })

path_lengths = sorted({len(p) for rec in cwq_pilot_planning for p in rec["predicted_paths"]})
print(f"Planned {len(cwq_pilot_planning)} CWQ questions. Relation-path lengths seen: {path_lengths}")

CWQ Planning (pilot):   0%|          | 0/50 [00:00<?, ?it/s]

Planned 50 CWQ questions. Relation-path lengths seen: [1, 2, 3, 4]


### BFS Sanity Check (Pilot)

In [45]:
mismatches = checked = 0
for rec in cwq_pilot_planning:
    graph = build_graph(rec["graph"])
    for entity in rec["q_entity"]:
        for rule in rec["predicted_paths"]:
            checked += 1
            if bfs_with_rule(graph, entity, rule) != bfs_with_rule_instrumented(graph, entity, rule)[0]:
                mismatches += 1

print(f"Checked {checked} calls. Mismatches: {mismatches}")
assert mismatches == 0, "Instrumented BFS diverges on CWQ -- STOP, do not trust downstream numbers."

Checked 258 calls. Mismatches: 0


### Retrieval (Pilot)

In [46]:
def run_retrieval(planning_recs, desc):
    out = []
    for rec in tqdm(planning_recs, desc=desc):
        graph = build_graph(rec["graph"])
        gold_answers = set(rec["a_entity"])
        plans = rec["predicted_paths"] if len(rec["predicted_paths"]) > 0 else [[]]
        for rule in plans:
            t0 = time.time()
            all_paths, hop_frontier_total = [], [0] * len(rule)
            if len(rule) > 0:
                for entity in rec["q_entity"]:
                    paths, hop_frontier = bfs_with_rule_instrumented(graph, entity, rule)
                    all_paths.extend(paths)
                    for h in range(len(rule)):
                        hop_frontier_total[h] += hop_frontier[h]
            retrieval_time = time.time() - t0
            reachable_gold = sorted(path_endpoints(all_paths) & gold_answers)
            out.append({
                "question_id": rec["id"], "question": rec["question"],
                "topic_entity": rec["q_entity"], "predicted_relation_path": rule,
                "hop_frontiers": hop_frontier_total,
                "retrieved_paths_count": len(all_paths),
                "gold_answers": sorted(gold_answers),
                "reachable_gold_answers": reachable_gold,
                "gold_reachable": len(reachable_gold) > 0,
                "retrieval_time_sec": retrieval_time,
                "planning_time_sec": rec["planning_time_sec"],
            })
    return out

cwq_pilot_retrieval = run_retrieval(cwq_pilot_planning, "CWQ Retrieval (pilot)")
df_cwq_pilot = pd.DataFrame(cwq_pilot_retrieval)
coverage = df_cwq_pilot.groupby("question_id")["gold_reachable"].any()
print(f"CWQ pilot coverage: {coverage.sum()} / {len(coverage)} = {coverage.mean()*100:.2f}%")

CWQ Retrieval (pilot):   0%|          | 0/50 [00:00<?, ?it/s]

CWQ pilot coverage: 36 / 50 = 72.00%


### Computing missed_qids_cwq

In [47]:
from collections import defaultdict

reachable_by_question_cwq = defaultdict(bool)
for rec in cwq_pilot_retrieval:
    reachable_by_question_cwq[rec["question_id"]] |= rec["gold_reachable"]

all_qids_cwq = {rec["question_id"] for rec in cwq_pilot_retrieval}
missed_qids_cwq = [qid for qid in all_qids_cwq if not reachable_by_question_cwq[qid]]

print(f"{len(missed_qids_cwq)} / {len(all_qids_cwq)} CWQ pilot questions missed")

14 / 50 CWQ pilot questions missed


### Revised Failure Analysis

In [48]:
from collections import deque, Counter

def find_gold_paths_official_graph(rec, max_hops=4, max_expansions=200000):
    G = build_graph(rec["graph"])
    gold = set(rec["a_entity"])
    found = []
    expansions = 0
    capped = False

    for start in rec["q_entity"]:
        queue = deque([(start, [], [start])])
        while queue:
            node, relations, entities = queue.popleft()

            if len(relations) > 0 and node in gold:
                found.append({
                    "relations": relations,
                    "entities": entities
                })

            if len(relations) == max_hops:
                continue
            if node not in G:
                continue

            for neighbor in G.neighbors(node):
                expansions += 1
                if expansions > max_expansions:
                    capped = True
                    return found, capped
                rel = G[node][neighbor]["relation"]
                queue.append((
                    neighbor,
                    relations + [rel],
                    entities + [neighbor]
                ))

    return found, capped


diagnosis_counts = Counter()

for qid in missed_qids_cwq:
    rec = next(r for r in cwq_pilot_planning if r["id"] == qid)   # <-- fixed: pilot planning list

    gold_paths, capped = find_gold_paths_official_graph(rec, max_hops=CWQ_MAX_HOPS)

    predicted = {
        tuple(p) for p in rec["predicted_paths"]
    }
    gold_relation_paths = {
        tuple(p["relations"]) for p in gold_paths
    }

    print("\n" + "="*100)
    print("QUESTION:", rec["question"])
    print("GOLD:", rec["a_entity"])

    print("\nPredicted:")
    for p in predicted:
        print(" ", " -> ".join(p))

    print("\nGold-supporting paths in OFFICIAL BUILT GRAPH:")
    for p in gold_paths[:10]:
        print(
            " ",
            " -> ".join(p["entities"]),
            "\n    ",
            " -> ".join(p["relations"])
        )

    if capped:
        diagnosis = "UNVERIFIED (hit expansion cap)"
    elif not gold_paths:
        diagnosis = "OFFICIAL GRAPH COVERAGE FAILURE"
    elif predicted.isdisjoint(gold_relation_paths):
        diagnosis = "PLANNER FAILURE"
    else:
        diagnosis = "TRUE RETRIEVAL ANOMALY"

    diagnosis_counts[diagnosis] += 1
    print("\nDIAGNOSIS:", diagnosis)

print("\n" + "="*100)
print("FAILURE BREAKDOWN:", diagnosis_counts)


QUESTION: Who holds the position of Prime Minister in the country which contains Dire Dawa?
GOLD: ['Hailemariam Desalegn']

Predicted:
  government.government_position_held.basic_title -> government.government_position_held.office_holder
  location.location.containedby -> people.person.nationality
  government.government_position_held.basic_title -> government.politician.government_positions_held

Gold-supporting paths in OFFICIAL BUILT GRAPH:

DIAGNOSIS: OFFICIAL GRAPH COVERAGE FAILURE

QUESTION: Which location in the Anadyr Timezone has the biggest population?
GOLD: ['India']

Predicted:
  time.time_zone.locations_in_this_time_zone
  location.location.time_zones
  location.location.containedby

Gold-supporting paths in OFFICIAL BUILT GRAPH:
  Anadyr Time Zone -> Asia -> India 
     location.location.time_zones -> location.location.containedby
  Anadyr Time Zone -> Day DST ends -> India Time Zone -> India 
     freebase.valuenotation.has_no_value -> freebase.valuenotation.has_no_valu

## CSV/JSON Save

In [49]:
import pandas as pd

json_path = os.path.join(CWQ_PILOT_DIR, "step1_baseline_cwq_pilot.json")
csv_path = os.path.join(CWQ_PILOT_DIR, "step1_baseline_cwq_pilot.csv")

with open(json_path, "w") as f:
    json.dump(cwq_pilot_retrieval, f, indent=2)

df_cwq_pilot = pd.DataFrame(cwq_pilot_retrieval)
df_cwq_pilot.to_csv(csv_path, index=False)

print("Saved:", json_path)
print("Saved:", csv_path)
df_cwq_pilot[["retrieved_paths_count", "gold_reachable", "retrieval_time_sec"]].describe()

Saved: /kaggle/working/step1_cwq_pilot/step1_baseline_cwq_pilot.json
Saved: /kaggle/working/step1_cwq_pilot/step1_baseline_cwq_pilot.csv


,retrieved_paths_count,retrieval_time_sec
count,150.000000,150.000000
mean,13.360000,0.000265
std,56.074289,0.000360
min,0.000000,0.000004
25%,0.000000,0.000031
50%,1.000000,0.000103
75%,7.000000,0.000296
max,630.000000,0.002367


### Worked Example

In [50]:
example = next(r for r in cwq_pilot_retrieval if len(r["predicted_relation_path"]) > 0)

print("Question:")
print(example["question"])
print()
print("Topic entity:")
print(example["topic_entity"])
print()
print("RoG relation plan:")
print(" -> ".join(example["predicted_relation_path"]))
print()
for i, count in enumerate(example["hop_frontiers"], start=1):
    print(f"Hop {i}:")
    print(f"{count} candidate entities")
    print()
print("Retrieved paths:")
print(example["retrieved_paths_count"])
print()
print("Gold answer reachable:")
print("Yes" if example["gold_reachable"] else "No")
print()
print("Retrieval time:")
print(f"{example['retrieval_time_sec']*1000:.1f} ms")

Question:
Lou Seal is the mascot for the team that last won the World Series when?

Topic entity:
['Lou Seal']

RoG relation plan:
sports.mascot.team -> sports.sports_championship_event.champion

Hop 1:
1 candidate entities

Hop 2:
1 candidate entities

Retrieved paths:
1

Gold answer reachable:
Yes

Retrieval time:
0.2 ms


### Final RoG Answer

In [51]:
from qa_prediction.build_qa_input import PromptBuilder

reasoning_prompter = PromptBuilder(
    os.path.join(REPO_DIR, "prompts", "llama2_predict.txt"),
    add_rule=True,
    maximun_token=4096 - 100,
    tokenize=lambda t: len(tokenizer.tokenize(t)),
)

by_question_cwq = {}
for rec in cwq_pilot_planning:
    by_question_cwq[rec["id"]] = rec

final_answers_cwq = {}
for qid, rec in tqdm(by_question_cwq.items(), desc="CWQ Reasoning (pilot)"):
    q_dict = {
        "question": rec["question"],
        "graph": rec["graph"],
        "q_entity": rec["q_entity"],
        "predicted_paths": rec["predicted_paths"],
        "choices": [],
    }
    prompt = reasoning_prompter.process_input(q_dict)
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(model.device)
    t0 = time.time()
    with torch.inference_mode():
        out = model.generate(input_ids=input_ids, max_new_tokens=512, do_sample=True)
    gen_time = time.time() - t0
    text = tokenizer.decode(out[0][input_ids.shape[1]:], skip_special_tokens=True).strip()
    final_answers_cwq[qid] = {"final_RoG_answer": text, "reasoning_time_sec": gen_time}

for rec in cwq_pilot_retrieval:
    rec.update(final_answers_cwq.get(rec["question_id"], {}))

with open(json_path, "w") as f:
    json.dump(cwq_pilot_retrieval, f, indent=2)
pd.DataFrame(cwq_pilot_retrieval).to_csv(csv_path, index=False)
print("Updated with final_RoG_answer and re-saved.")

CWQ Reasoning (pilot):   0%|          | 0/50 [00:00<?, ?it/s]

Updated with final_RoG_answer and re-saved.


### Verify the Decode Patch (full set)

In [52]:
cwq_default_decode_records = []
for sample in tqdm(cwq_pilot, desc="CWQ Verify (default decode)"):
    input_text = prompter.format(instruction=INSTRUCTION, message=sample["question"])
    input_ids = tokenizer.encode(input_text, return_tensors="pt").to(model.device)
    with torch.inference_mode():
        output = model.generate(
            input_ids=input_ids, num_beams=N_BEAM, num_return_sequences=N_BEAM,
            early_stopping=False, do_sample=True, return_dict_in_generate=True,
            output_scores=True, max_new_tokens=100,
        )
    raw_sequences = output.sequences[:, input_ids.shape[1]:]
    default_text = [_original_decode(seq, skip_special_tokens=True).strip() for seq in raw_sequences]
    default_paths = parse_prediction(default_text)
    cwq_default_decode_records.append({"id": sample["id"], "predicted_paths": default_paths})

by_id_default_cwq = {r["id"]: r["predicted_paths"] for r in cwq_default_decode_records}
diffs = 0
examples_shown = 0
for rec in cwq_pilot_planning:
    patched = rec["predicted_paths"]
    default = by_id_default_cwq.get(rec["id"], [])
    if patched != default:
        diffs += 1
        if examples_shown < 5:
            print(f"\nQID {rec['id']}  --  {rec['question']}")
            print("  patched (sp_model.decode): ", patched)
            print("  default (tokenizer.decode):", default)
            examples_shown += 1

print(f"\n{diffs} / {len(cwq_pilot_planning)} questions differ between patched and default decode.")

CWQ Verify (default decode):   0%|          | 0/50 [00:00<?, ?it/s]


0 / 50 questions differ between patched and default decode.


### Runtime Estimate

In [53]:
mean_plan_cwq = df_cwq_pilot["planning_time_sec"].mean()
mean_retr_cwq = df_cwq_pilot["retrieval_time_sec"].mean()
n_cwq_full = 3531  # official CWQ test size (RoG paper, Table 6)

est_plan_sec = mean_plan_cwq * n_cwq_full
est_retr_sec = mean_retr_cwq * n_cwq_full
est_total_hr = (est_plan_sec + est_retr_sec) / 3600

print(f"Mean planning time/question:  {mean_plan_cwq:.2f} s")
print(f"Mean retrieval time/question: {mean_retr_cwq:.3f} s")
print(f"Projected for {n_cwq_full} questions:")
print(f"  Planning:  ~{est_plan_sec/60:.1f} min")
print(f"  Retrieval: ~{est_retr_sec/60:.1f} min")
print(f"  Total:     ~{est_total_hr:.2f} GPU-hours (planning+retrieval only, not the final-answer pass)")

Mean planning time/question:  2.06 s
Mean retrieval time/question: 0.000 s
Projected for 3531 questions:
  Planning:  ~121.0 min
  Retrieval: ~0.0 min
  Total:     ~2.02 GPU-hours (planning+retrieval only, not the final-answer pass)


### Full-Scale Config

In [54]:
CWQ_FULL_DIR = "/kaggle/working/step1_cwq_full"
os.makedirs(CWQ_FULL_DIR, exist_ok=True)
# === RESTORE CWQ checkpoints from last session ===
if PREV_RUN_DIR:
    prev_dir = os.path.join(PREV_RUN_DIR, "step1_cwq_full")
    if os.path.exists(prev_dir):
        for fname in os.listdir(prev_dir):
            dst = os.path.join(CWQ_FULL_DIR, fname)
            if not os.path.exists(dst):
                shutil.copy2(os.path.join(prev_dir, fname), dst)
                print("Restored:", fname)
    else:
        print("No prior CWQ full-run folder in PREV_RUN_DIR.")
else:
    print("PREV_RUN_DIR not set — nothing to restore.")
CWQ_PLANNING_CKPT = os.path.join(CWQ_FULL_DIR, "planning_cwq_full.jsonl")
CWQ_REASON_CKPT = os.path.join(CWQ_FULL_DIR, "reasoning_cwq_full.jsonl")

SEED = 42
torch.manual_seed(SEED)

cwq_full_test_all = load_dataset(CWQ_DATASET_NAME, split=CWQ_SPLIT)
print(f"Full CWQ test set: {len(cwq_full_test_all)} questions")

Full CWQ test set: 3531 questions


### Checkpointed Planning (full set)

In [55]:
cwq_planning_done = load_checkpoint(CWQ_PLANNING_CKPT)
print(f"Resuming: {len(cwq_planning_done)} CWQ questions already planned.")

with open(CWQ_PLANNING_CKPT, "a") as fout:
    for sample in tqdm(cwq_full_test_all, desc="Planning (CWQ full)"):
        if sample["id"] in cwq_planning_done:
            continue
        input_text = prompter.format(instruction=INSTRUCTION, message=sample["question"])
        t0 = time.time()
        raw_output = generate_seq(model, input_text, tokenizer, num_beam=N_BEAM, do_sample=True, max_new_tokens=100)
        planning_time = time.time() - t0
        rel_paths = parse_prediction(raw_output["paths"])
        rec = {
            "id": sample["id"], "question": sample["question"],
            "q_entity": sample["q_entity"], "a_entity": sample["a_entity"],
            "graph": sample["graph"], "predicted_paths": rel_paths,
            "planning_time_sec": planning_time,
        }
        fout.write(json.dumps(rec) + "\n")
        fout.flush()
        cwq_planning_done[sample["id"]] = rec

cwq_planning_records_full = list(cwq_planning_done.values())
print(f"Planning complete: {len(cwq_planning_records_full)} / {len(cwq_full_test_all)} questions.")

Resuming: 3531 CWQ questions already planned.


Planning (CWQ full):   0%|          | 0/3531 [00:00<?, ?it/s]

Planning complete: 3531 / 3531 questions.


### Retrieval (full set)

In [56]:
from tqdm.auto import tqdm
cwq_retrieval_records_full = []
for rec in tqdm(cwq_planning_records_full, desc="Retrieval (CWQ full)"):
    graph = build_graph(rec["graph"])
    gold_answers = set(rec["a_entity"])
    plans = rec["predicted_paths"] if len(rec["predicted_paths"]) > 0 else [[]]
    for rule in plans:
        t0 = time.time()
        all_paths = []
        hop_frontier_total = [0] * len(rule)
        if len(rule) > 0:
            for entity in rec["q_entity"]:
                paths, hop_frontier = bfs_with_rule_instrumented(graph, entity, rule)
                all_paths.extend(paths)
                for h in range(len(rule)):
                    hop_frontier_total[h] += hop_frontier[h]
        retrieval_time = time.time() - t0
        reachable_gold = sorted(path_endpoints(all_paths) & gold_answers)
        cwq_retrieval_records_full.append({
            "question_id": rec["id"], "question": rec["question"],
            "topic_entity": rec["q_entity"], "predicted_relation_path": rule,
            "hop_frontiers": hop_frontier_total,
            "retrieved_paths_count": len(all_paths),
            "gold_answers": sorted(gold_answers),
            "reachable_gold_answers": reachable_gold,
            "gold_reachable": len(reachable_gold) > 0,
            "retrieval_time_sec": retrieval_time,
            "planning_time_sec": rec["planning_time_sec"],
        })

print(f"{len(cwq_retrieval_records_full)} retrieval records (CWQ full) collected.")

Retrieval (CWQ full):   0%|          | 0/3531 [00:00<?, ?it/s]

10566 retrieval records (CWQ full) collected.


### Question-Level Coverage

In [57]:
df_cwq_full = pd.DataFrame(cwq_retrieval_records_full)
question_coverage_cwq_full = df_cwq_full.groupby("question_id")["gold_reachable"].any()
print("Questions evaluated:", len(question_coverage_cwq_full))
print("Questions reaching >=1 gold answer:", question_coverage_cwq_full.sum())
print("Question-level retrieval coverage: %.2f%%" % (question_coverage_cwq_full.mean() * 100))

Questions evaluated: 3531
Questions reaching >=1 gold answer: 2422
Question-level retrieval coverage: 68.59%


### Aggregated Failure Analysis

In [58]:
from collections import Counter

reachable_by_q_cwq_full = defaultdict(bool)
for rec in cwq_retrieval_records_full:
    reachable_by_q_cwq_full[rec["question_id"]] |= rec["gold_reachable"]

all_qids_cwq_full = {rec["question_id"] for rec in cwq_retrieval_records_full}
missed_qids_cwq_full = [q for q in all_qids_cwq_full if not reachable_by_q_cwq_full[q]]
print(f"{len(missed_qids_cwq_full)} / {len(all_qids_cwq_full)} questions missed")

cwq_planning_by_id_full = {r["id"]: r for r in cwq_planning_records_full}
diagnosis_counts_cwq = Counter()
diagnosis_by_qid_cwq = {}
SAMPLE_PRINT = 5
printed = 0

for qid in tqdm(missed_qids_cwq_full, desc="Failure analysis (CWQ full)"):
    rec = cwq_planning_by_id_full[qid]
    gold_paths, capped = find_gold_paths_official_graph(rec, max_hops=CWQ_MAX_HOPS)
    predicted = {tuple(p) for p in rec["predicted_paths"]}
    gold_relation_paths = {tuple(p["relations"]) for p in gold_paths}

    if capped:
        diagnosis = "UNVERIFIED (hit expansion cap)"
    elif not gold_paths:
        diagnosis = "OFFICIAL GRAPH COVERAGE FAILURE"
    elif predicted.isdisjoint(gold_relation_paths):
        diagnosis = "PLANNER FAILURE"
    else:
        diagnosis = "TRUE RETRIEVAL ANOMALY"

    diagnosis_counts_cwq[diagnosis] += 1
    diagnosis_by_qid_cwq[qid] = diagnosis

    if printed < SAMPLE_PRINT:
        print("\n" + "="*100)
        print("QUESTION:", rec["question"])
        print("DIAGNOSIS:", diagnosis)
        printed += 1

print("\n--- Failure breakdown ---")
for k, v in diagnosis_counts_cwq.most_common():
    print(f"{k}: {v}  ({100*v/max(len(missed_qids_cwq_full),1):.2f}%)")

1109 / 3531 questions missed


Failure analysis (CWQ full):   0%|          | 0/1109 [00:00<?, ?it/s]


QUESTION: What Government position holder fought in the battle of Vicksburg?
DIAGNOSIS: UNVERIFIED (hit expansion cap)

QUESTION: Which Attorney general fought in the battle of Vicksburg?
DIAGNOSIS: UNVERIFIED (hit expansion cap)

QUESTION: Who does Queens hair on a film that James Jordan was a crew member on?
DIAGNOSIS: PLANNER FAILURE

QUESTION: For which teams did the father of the performer in "NFL Super Bowl XLIV Champions: New Orleans Saints" play?
DIAGNOSIS: UNVERIFIED (hit expansion cap)

QUESTION: Who is the prime minister of the country that is the major exports of coffee and tea?
DIAGNOSIS: OFFICIAL GRAPH COVERAGE FAILURE

--- Failure breakdown ---
UNVERIFIED (hit expansion cap): 663  (59.78%)
OFFICIAL GRAPH COVERAGE FAILURE: 352  (31.74%)
PLANNER FAILURE: 94  (8.48%)


### Full-Scale Output

In [59]:
cwq_full_json_path = os.path.join(CWQ_FULL_DIR, "step1_baseline_cwq_full.json")
cwq_full_csv_path = os.path.join(CWQ_FULL_DIR, "step1_baseline_cwq_full.csv")

with open(cwq_full_json_path, "w") as f:
    json.dump(cwq_retrieval_records_full, f, indent=2)
pd.DataFrame(cwq_retrieval_records_full).to_csv(cwq_full_csv_path, index=False)
print("Saved:", cwq_full_json_path)
print("Saved:", cwq_full_csv_path)

Saved: /kaggle/working/step1_cwq_full/step1_baseline_cwq_full.json
Saved: /kaggle/working/step1_cwq_full/step1_baseline_cwq_full.csv


### Final RoG Answer (Full Planning)

In [60]:
final_answers_cwq_full = {}
with open(CWQ_REASON_CKPT, "a") as fout:
    reasoning_done_cwq = load_checkpoint(CWQ_REASON_CKPT)
    print(f"Resuming: {len(reasoning_done_cwq)} CWQ answers already generated.")

    for qid, rec in tqdm(cwq_planning_by_id_full.items(), desc="Reasoning (CWQ full)"):
        if qid in reasoning_done_cwq:
            continue
        q_dict = {
            "question": rec["question"], "graph": rec["graph"],
            "q_entity": rec["q_entity"], "predicted_paths": rec["predicted_paths"],
            "choices": [],
        }
        prompt = reasoning_prompter.process_input(q_dict)
        input_ids = tokenizer.encode(prompt, return_tensors="pt").to(model.device)
        t0 = time.time()
        with torch.inference_mode():
            out = model.generate(input_ids=input_ids, max_new_tokens=512, do_sample=True)
        gen_time = time.time() - t0
        text = tokenizer.decode(out[0][input_ids.shape[1]:], skip_special_tokens=True).strip()
        out_rec = {"id": qid, "final_RoG_answer": text, "reasoning_time_sec": gen_time}
        fout.write(json.dumps(out_rec) + "\n")
        fout.flush()
        reasoning_done_cwq[qid] = out_rec

for rec in cwq_retrieval_records_full:
    ans = reasoning_done_cwq.get(rec["question_id"])
    if ans:
        rec.update({"final_RoG_answer": ans["final_RoG_answer"], "reasoning_time_sec": ans["reasoning_time_sec"]})

with open(cwq_full_json_path, "w") as f:
    json.dump(cwq_retrieval_records_full, f, indent=2)
pd.DataFrame(cwq_retrieval_records_full).to_csv(cwq_full_csv_path, index=False)
print("Updated with final_RoG_answer and re-saved.")

Resuming: 3531 CWQ answers already generated.


Reasoning (CWQ full):   0%|          | 0/3531 [00:00<?, ?it/s]

Updated with final_RoG_answer and re-saved.


### Confirm the Reasoning Loop

In [61]:
missing_cwq = [qid for qid in cwq_planning_by_id_full if qid not in reasoning_done_cwq]
print(f"{len(reasoning_done_cwq)} / {len(cwq_planning_by_id_full)} CWQ questions have a final answer.")
if missing_cwq:
    print(f"{len(missing_cwq)} still missing -- re-run the reasoning cell above (it resumes automatically).")
else:
    print("CWQ reasoning complete. Safe to proceed.")

3531 / 3531 CWQ questions have a final answer.
CWQ reasoning complete. Safe to proceed.


### Memory Cache for RoG-cwq


In [62]:
# === MEMORY: drop cached subgraphs now that CWQ reasoning is done ===
for rec in cwq_planning_records_full:
    rec.pop("graph", None)
gc.collect()
print("Dropped cached subgraphs from cwq_planning_records_full to free RAM.")

Dropped cached subgraphs from cwq_planning_records_full to free RAM.


### Clean Per-Quesstion Table

In [63]:
retrieval_time_by_qid_cwq = defaultdict(float)
gold_reachable_by_qid_cwq = defaultdict(bool)
for rec in cwq_retrieval_records_full:
    retrieval_time_by_qid_cwq[rec["question_id"]] += rec["retrieval_time_sec"]
    gold_reachable_by_qid_cwq[rec["question_id"]] |= rec["gold_reachable"]

rows_cwq = []
for qid, prec in cwq_planning_by_id_full.items():
    rrow = reasoning_done_cwq.get(qid, {})
    rows_cwq.append({
        "id": qid,
        "question": prec["question"],
        "gold_answers": prec["a_entity"],
        "predicted_paths": prec["predicted_paths"],
        "planning_time_sec": prec["planning_time_sec"],
        "retrieval_time_sec": retrieval_time_by_qid_cwq.get(qid, 0.0),
        "reasoning_time_sec": rrow.get("reasoning_time_sec"),
        "final_RoG_answer": rrow.get("final_RoG_answer"),
        "gold_reachable": gold_reachable_by_qid_cwq.get(qid, False),
    })

cwq_per_question = pd.DataFrame(rows_cwq)
n_cwq = len(cwq_per_question)
assert cwq_per_question["reasoning_time_sec"].notna().all(), "Some CWQ questions missing reasoning -- finish that loop first."
print(f"CWQ per-question baseline table: {n_cwq} questions.")

CWQ per-question baseline table: 3531 questions.


### Full Question-Per Table

In [64]:
pd.set_option("display.max_colwidth", 100)

display_cols = [
    "id", "question", "gold_answers", "predicted_paths",
    "planning_time_sec", "retrieval_time_sec", "reasoning_time_sec",
    "final_RoG_answer", "gold_reachable"
]

cwq_per_question_display = cwq_per_question[display_cols].rename(columns={"id": "question_id"})
cwq_per_question_display

,question_id,question,gold_answers,predicted_paths,planning_time_sec,retrieval_time_sec,reasoning_time_sec,final_RoG_answer,gold_reachable
0,WebQTest-832_c334509bb5e02cacae1ba2e80c176499,Lou Seal is the mascot for the team that last won the World Series when?,[2014 World Series],"[[sports.mascot.team, sports.sports_championship_event.champion], [sports.mascot.team, sports.sp...",2.251861,0.000374,1.090070,2014 World Series\n2012 World Series,True
1,WebQTrn-1259_1997cb4922db71983be26e6a509950f4,"Where did the ""Country Nation World Tour"" concert artist go to college?",[Belmont University],"[[music.concert_tour.artist, people.person.education, education.education.institution], [music.a...",2.370586,0.000483,0.446718,Berklee College of Music,False
2,WebQTest-1384_744a496b907e407b16bc5d7c197dc3f0,What is the predominant religion where the leader is Ovadia Yosef?,[Judaism],"[[people.person.religion], [people.person.nationality, base.argumentmaps.thing_of_disputed_value...",2.165733,0.000606,0.509523,Judaism,True
3,WebQTrn-241_dfb6c97ac9bf2f0ac07f27dd80f9edc2,What country bordering France contains an airport that serves Nijmegen?,[Germany],"[[olympics.olympic_participating_country.olympics_participated_in, olympics.olympic_participatin...",3.910672,0.005167,3.274526,Germany,True
4,WebQTrn-1077_f4a9e5f1e0dcfb82cbadf4771eda7bb5,The national anthem Afghan National Anthem is from the country which practices what religions?,"[Shia Islam, Sunni Islam]","[[music.composition.language, language.human_language.countries_spoken_in], [music.composition.l...",1.790218,0.000139,0.832482,Christianity\nBuddhism\nIslam,False
...,...,...,...,...,...,...,...,...,...
3526,WebQTrn-1938_0e945cac8043fe5af615e4b2f0ddac8f,What is the type of government practiced in the country where the Israeli Lira is used?,[Parliamentary system],"[[finance.currency.countries_formerly_used, government.form_of_government.countries], [finance.c...",2.235173,0.001682,0.282712,Parliamentary system,True
3527,WebQTest-989_20bb2e223d83caf91ca75a04b854377c,"What event with less than 30,000 casualties happened at Dunkirk in WW2?",[Battle of Dunkirk],"[[military.military_conflict.casualties, military.casualties.military_conflict], [military.milit...",2.387669,0.000491,0.680226,Raid on Dunkirk,False
3528,WebQTrn-1722_aca40552e6778874c40c75071d819a54,North America is where Bahamas Creole English Language is spoken belong to.?,[North America],"[[location.location.containedby], [language.human_language.region], [location.country.languages_...",0.910697,0.000039,0.466975,Bahamas\nAmericas,False
3529,WebQTrn-3543_bbb0c8aa3a2941db5bf85e7557241fda,"Find the country with the ISO number of 736 that that imports from Japan, what is the name of th...",[Sudan],"[[base.aareas.schema.administrative_area.administrative_area_type, base.aareas.schema.administra...",3.395276,0.002277,2.505633,Kiribati,True


### Saved as CSV and JSON

In [65]:
cwq_csv_out = os.path.join(CWQ_FULL_DIR, "cwq_per_question_baseline.csv")
cwq_json_out = os.path.join(CWQ_FULL_DIR, "cwq_per_question_baseline.json")

cwq_per_question_display.to_csv(cwq_csv_out, index=False)
cwq_per_question_display.to_json(cwq_json_out, orient="records", indent=2)

print("Saved:", cwq_csv_out)
print("Saved:", cwq_json_out)

Saved: /kaggle/working/step1_cwq_full/cwq_per_question_baseline.csv
Saved: /kaggle/working/step1_cwq_full/cwq_per_question_baseline.json


### Timing Statistics

In [66]:
def total_avg_cwq(col):
    total = cwq_per_question[col].sum()
    return total, total / n_cwq

plan_total_cwq, plan_avg_cwq = total_avg_cwq("planning_time_sec")
retr_total_cwq, retr_avg_cwq = total_avg_cwq("retrieval_time_sec")
reas_total_cwq, reas_avg_cwq = total_avg_cwq("reasoning_time_sec")
e2e_total_cwq = plan_total_cwq + retr_total_cwq + reas_total_cwq
e2e_avg_cwq = e2e_total_cwq / n_cwq

print(f"Total planning time:   {plan_total_cwq:9.1f} s  ({plan_total_cwq/60:7.2f} min)   avg/question: {plan_avg_cwq:.3f} s")
print(f"Total retrieval time:  {retr_total_cwq:9.1f} s  ({retr_total_cwq/60:7.2f} min)   avg/question: {retr_avg_cwq:.4f} s")
print(f"Total reasoning time:  {reas_total_cwq:9.1f} s  ({reas_total_cwq/60:7.2f} min)   avg/question: {reas_avg_cwq:.3f} s")
print(f"Total end-to-end time: {e2e_total_cwq:9.1f} s  ({e2e_total_cwq/60:7.2f} min)   avg/question: {e2e_avg_cwq:.3f} s")

Total planning time:      7044.8 s  ( 117.41 min)   avg/question: 1.995 s
Total retrieval time:        3.1 s  (   0.05 min)   avg/question: 0.0009 s
Total reasoning time:     6027.2 s  ( 100.45 min)   avg/question: 1.707 s
Total end-to-end time:   13075.1 s  ( 217.92 min)   avg/question: 3.703 s


### Retrieval Coverage and Official Hit/Hits@1/F1

In [67]:
import sys
sys.path.append(os.path.join(REPO_DIR, "src"))
from qa_prediction.evaluate_results import eval_acc, eval_hit, eval_f1

n_covered_cwq = int(cwq_per_question["gold_reachable"].sum())
coverage_pct_cwq = 100 * n_covered_cwq / n_cwq

hit_list_cwq, acc_list_cwq, f1_list_cwq, prec_list_cwq, rec_list_cwq = [], [], [], [], []
for _, row in cwq_per_question.iterrows():
    prediction = [p for p in row["final_RoG_answer"].split("\n") if p.strip()]
    prediction_str = " ".join(prediction)
    f1, precision, recall = eval_f1(prediction, row["gold_answers"])
    hit_list_cwq.append(eval_hit(prediction_str, row["gold_answers"]))
    acc_list_cwq.append(eval_acc(prediction_str, row["gold_answers"]))
    f1_list_cwq.append(f1); prec_list_cwq.append(precision); rec_list_cwq.append(recall)

hits1_pct_cwq = 100 * sum(hit_list_cwq) / n_cwq
f1_pct_cwq = 100 * sum(f1_list_cwq) / n_cwq

print(f"Questions reaching >=1 gold: {n_covered_cwq}")
print(f"Question-level retrieval coverage: {coverage_pct_cwq:.2f}%")
print(f"Hit/Hits@1: {hits1_pct_cwq:.2f}%")
print(f"F1: {f1_pct_cwq:.2f}%")

Questions reaching >=1 gold: 2422
Question-level retrieval coverage: 68.59%
Hit/Hits@1: 61.23%
F1: 54.32%


### Freezing Baseline

In [68]:
cwq_baseline_summary = f"""Dataset: CWQ
Test questions: {n_cwq}

Retrieval:
  Questions reaching >=1 gold: {n_covered_cwq}
  Question-level retrieval coverage: {coverage_pct_cwq:.2f}%

Timing:
  Total planning time:   {plan_total_cwq:.1f} s ({plan_total_cwq/60:.2f} min)   Avg/question: {plan_avg_cwq:.3f} s
  Total retrieval time:  {retr_total_cwq:.1f} s ({retr_total_cwq/60:.2f} min)   Avg/question: {retr_avg_cwq:.4f} s
  Total reasoning time:  {reas_total_cwq:.1f} s ({reas_total_cwq/60:.2f} min)   Avg/question: {reas_avg_cwq:.3f} s
  Total end-to-end time: {e2e_total_cwq:.1f} s ({e2e_total_cwq/60:.2f} min)   Avg/question: {e2e_avg_cwq:.3f} s

Final QA:
  Hit/Hits@1: {hits1_pct_cwq:.2f}%
  F1: {f1_pct_cwq:.2f}%
"""
print(cwq_baseline_summary)

with open(os.path.join(CWQ_FULL_DIR, "cwq_baseline_summary.txt"), "w") as f:
    f.write(cwq_baseline_summary)

cwq_per_question.to_json(os.path.join(CWQ_FULL_DIR, "cwq_baseline_predictions.json"), orient="records", indent=2)
cwq_per_question.to_csv(os.path.join(CWQ_FULL_DIR, "cwq_baseline_predictions.csv"), index=False)
print("CWQ Baseline frozen.")

Dataset: CWQ
Test questions: 3531

Retrieval:
  Questions reaching >=1 gold: 2422
  Question-level retrieval coverage: 68.59%

Timing:
  Total planning time:   7044.8 s (117.41 min)   Avg/question: 1.995 s
  Total retrieval time:  3.1 s (0.05 min)   Avg/question: 0.0009 s
  Total reasoning time:  6027.2 s (100.45 min)   Avg/question: 1.707 s
  Total end-to-end time: 13075.1 s (217.92 min)   Avg/question: 3.703 s

Final QA:
  Hit/Hits@1: 61.23%
  F1: 54.32%

CWQ Baseline frozen.
